# Inspect Generation Samples

Set `PROMPT_IDX` and run the notebook to inspect the fixed prompt, baseline output, Value-Based outputs, and Residual(attn) outputs from matched generation JSONL files.

In [ ]:
import json
import re
from pathlib import Path

ROOT = Path("../representation-analysis/outputs/generate").resolve()
PROMPT_IDX = 46
CANDIDATE_PROMPT_IDXS = [46, 27, 3, 8]
MAX_CHARS = 420
KEEP_THINK_TAGS = False
INCLUDE_MLP_BOTH = False
INCLUDE_COUNTER_SETTINGS = False

PARA_SETTINGS = ["-1", "0", "2"]
PERP_SETTINGS = ["0", "0.5", "1.5"]

ROOT

In [ ]:
def load_prompt_row(path: Path, prompt_idx: int):
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            if row.get("prompt_idx") == prompt_idx:
                return row
    raise KeyError(f"prompt_idx={prompt_idx} not found in {path}")


def clean_text(text: str, keep_think_tags: bool = KEEP_THINK_TAGS) -> str:
    if not keep_think_tags:
        text = text.replace("<think>", "").replace("</think>", "")
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", text).strip()


def maybe_clip(text: str, max_chars: int = MAX_CHARS) -> str:
    if max_chars is None or len(text) <= max_chars:
        return text
    return text[: max_chars - 4].rstrip() + " ..."


def get_nested(row, *keys):
    value = row["generations"]
    for key in keys:
        value = value[key]
    return value


def perp_file_scale(scale: str) -> str:
    return "0.0" if scale == "0" else scale


def collect_samples(prompt_idx: int = PROMPT_IDX):
    rows = []
    baseline_path = ROOT / "para" / "generate_para_scale_1.jsonl"
    baseline_row = load_prompt_row(baseline_path, prompt_idx)
    prompt = clean_text(baseline_row["prompt"], keep_think_tags=True)
    baseline = clean_text(get_nested(baseline_row, "residual_output", "none"))
    rows.append({"setting": "Prompt", "output": prompt})
    rows.append({"setting": "Baseline | no edit", "output": baseline})

    for scale in PARA_SETTINGS:
        row = load_prompt_row(ROOT / "para" / f"generate_para_scale_{scale}.jsonl", prompt_idx)
        rows.append({
            "setting": f"Value-Based | s_parallel={scale}",
            "output": clean_text(get_nested(row, "xsa_middle_multihead", "attn")),
        })
        if INCLUDE_COUNTER_SETTINGS:
            rows.append({
                "setting": f"Residual(attn) | s_parallel={scale}",
                "output": clean_text(get_nested(row, "residual_output", "attn")),
            })
        if INCLUDE_MLP_BOTH:
            for branch in ("mlp", "both"):
                rows.append({
                    "setting": f"Residual({branch}) | s_parallel={scale}",
                    "output": clean_text(get_nested(row, "residual_output", branch)),
                })

    for scale in PERP_SETTINGS:
        row = load_prompt_row(
            ROOT / "perp" / f"generate_para_scale_1.0_perp_scale_{perp_file_scale(scale)}.jsonl",
            prompt_idx,
        )
        rows.append({
            "setting": f"Residual(attn) | s_parallel=1.0, s_perp={scale}",
            "output": clean_text(get_nested(row, "residual_output", "attn")),
        })
        if INCLUDE_COUNTER_SETTINGS:
            rows.append({
                "setting": f"Value-Based | s_parallel=1.0, s_perp={scale}",
                "output": clean_text(get_nested(row, "xsa_middle_multihead", "attn")),
            })
        if INCLUDE_MLP_BOTH:
            for branch in ("mlp", "both"):
                rows.append({
                    "setting": f"Residual({branch}) | s_parallel=1.0, s_perp={scale}",
                    "output": clean_text(get_nested(row, "residual_output", branch)),
                })
    return rows


samples = collect_samples(PROMPT_IDX)
len(samples)


In [ ]:
for item in samples:
    print(f"\n### {item['setting']}")
    print(maybe_clip(item["output"]))

In [ ]:
# Optional: render as a compact dataframe for quick scanning.
import pandas as pd

df = pd.DataFrame(samples)
df["excerpt"] = df["output"].map(lambda x: maybe_clip(x))
df[["setting", "excerpt"]]

In [ ]:
# Optional: generate LaTeX table rows after selecting settings.
SELECTED = [
    "Baseline | no edit",
    "Value-Based | s_parallel=-1",
    "Value-Based | s_parallel=0",
    "Value-Based | s_parallel=2",
    "Residual(attn) | s_parallel=1.0, s_perp=0",
    "Residual(attn) | s_parallel=1.0, s_perp=0.5",
    "Residual(attn) | s_parallel=1.0, s_perp=1.5",
]

def latex_escape(text: str) -> str:
    repl = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(repl.get(ch, ch) for ch in text)


by_setting = {item["setting"]: item["output"] for item in samples}
for setting in SELECTED:
    output = latex_escape(maybe_clip(by_setting[setting], 260))
    print(f"{latex_escape(setting)} &\n\\emph{{{output}}} \\\\")
